# Image Modality Experiment: Do Listing Photos Help?

Standalone experiment -- does **not** touch `main.py` / `notebooks/pipeline.ipynb`.

Neither `calendar.csv.gz` nor listing photos have been used anywhere in the
pipeline so far, even though `picture_url` is available in the raw data and
the course brief lists images as a suggested modality. This notebook:

1. Samples ~3,000 listings with a valid `picture_url` (downloading and
   embedding all ~40k photos isn't necessary to answer "does this help at all").
2. Downloads each photo and extracts a 512-d embedding from a frozen,
   ImageNet-pretrained ResNet18 (no fine-tuning -- just a feature extractor).
3. Reduces the embedding to 30 PCA components (fit on the training split only,
   to avoid leakage) to keep the feature count sane relative to the sample size.
4. Compares XGBoost with vs. without the image features, **on the same
   ~3,000-listing subset** (not the full 39k dataset) so the comparison is fair.

This is a feasibility test, not a proposal to always download 40k photos on
every pipeline run -- that would only make sense as a deliberate, cached,
opt-in step if it turns out to help.


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from src.config import CLEANED_TARGET_PATH, RANDOM_STATE, RAW_LISTINGS_PATH, TARGET_COLUMN, TEST_SIZE
from src.data.clean import clean_missing_target
from src.evaluate import print_metrics_log_target
from src.features.build_features import build_feature_matrix
from src.features.categorical import fit_categorical_encoders, transform_categorical_features
from src.features.image import build_image_embeddings, download_images_concurrent
from src.models.tree_models import train_xgboost


## Step 1: Build the same feature set as the main pipeline (numeric + categorical)

In [2]:
clean_missing_target(RAW_LISTINGS_PATH, CLEANED_TARGET_PATH, target_column=TARGET_COLUMN)
df = pd.read_csv(CLEANED_TARGET_PATH)
df = build_feature_matrix(df)
df.shape


Reading /Users/jnanadeep/Desktop/airbnb-price-prediction/data/raw/listings.csv...



--- Processing Summary ---
Initial row count:          40769
Rows with missing 'price': 953 (Removed)
Remaining row count:        39816
Cleaned data saved to:      /Users/jnanadeep/Desktop/airbnb-price-prediction/data/processed/airbnb_rio_cleaned_target.csv



Dropping columns with >90.0% missing values:
 - neighborhood_overview
 - host_since
 - host_response_time
 - host_response_rate
 - host_acceptance_rate
 - host_thumbnail_url
 - host_neighbourhood
 - host_total_listings_count
 - host_verifications
 - neighbourhood
 - neighbourhood_group_cleansed
 - calendar_updated
 - license
 - instant_bookable

Imputed 25 numerical columns using their median value.
--- Running Feature Selection ---
Original unique columns: 76
Filtered columns retained: 24
--- Running Dynamic Outlier Removal ---
Calculated 99.0th percentile threshold: $6,999.13
Removed 399 listing(s) outside the range [$10 - $6,999.13].
New maximum price in dataset: $6,990.00


(39417, 27)

## Step 2: Sample ~3,000 listings with a photo and download them

In [3]:
raw = pd.read_csv(RAW_LISTINGS_PATH, usecols=["id", "picture_url"]).dropna(subset=["picture_url"])
raw = raw[raw["id"].isin(df["id"])]

SAMPLE_SIZE = 3000
sample = raw.sample(min(SAMPLE_SIZE, len(raw)), random_state=RANDOM_STATE)

print(f"Downloading {len(sample)} photos...")
images_by_url = download_images_concurrent(sample["picture_url"].tolist(), max_workers=32)
print(f"Downloaded {len(images_by_url)}/{len(sample)} photos successfully")

url_to_id = dict(zip(sample["picture_url"], sample["id"]))
images_by_id = {url_to_id[url]: img for url, img in images_by_url.items()}


/opt/anaconda3/lib/python3.13/site-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (102257750 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Downloaded 2894/3000 photos successfully


## Step 3: Extract ResNet18 embeddings

In [4]:
embeddings = build_image_embeddings(images_by_id)
print("Embeddings shape:", embeddings.shape)
embeddings.head()


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/jnanadeep/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

  3%|▎         | 1.12M/44.7M [00:00<00:04, 11.2MB/s]

 11%|█         | 4.88M/44.7M [00:00<00:01, 27.3MB/s]

 27%|██▋       | 11.9M/44.7M [00:00<00:00, 48.0MB/s]

...

 83%|████████▎ | 37.0M/44.7M [00:00<00:00, 56.8MB/s]

 98%|█████████▊| 43.8M/44.7M [00:00<00:00, 60.8MB/s]

100%|██████████| 44.7M/44.7M [00:00<00:00, 55.2MB/s]

Embeddings shape: (2894, 512)


,img_0,img_1,img_2,img_3,img_4,img_5,img_6,img_7,img_8,img_9,...,img_502,img_503,img_504,img_505,img_506,img_507,img_508,img_509,img_510,img_511
id,,,,,,,,,,,,,,,,,,,,,
10065255,0.374086,1.191883,1.610998,1.156109,0.621563,2.093076,1.480997,0.298976,0.610931,1.060832,...,1.182949,0.073593,1.708104,0.148252,1.249831,0.535393,0.647378,1.561801,0.654671,0.420013
1617437679345701094,2.179412,2.282512,1.129502,0.845466,0.184830,3.013517,0.036449,0.611477,0.881514,0.146906,...,0.992894,0.288755,0.098435,0.104418,0.008190,1.012680,1.199376,0.427619,1.935228,1.072548
1053428487620174682,0.930172,0.488831,1.152528,0.147057,0.303938,0.704983,0.400857,0.480195,3.090251,1.407352,...,0.658192,1.200974,0.414747,0.976559,0.241832,0.232855,1.364564,2.633239,1.027038,0.469425
1153859206915993936,0.694646,0.360440,0.655743,0.688322,0.310879,0.345377,0.843014,0.811593,1.667708,1.960233,...,0.256310,0.639012,0.658728,0.058084,0.325239,0.599844,1.804896,2.994455,0.459858,0.862982
3231692,0.780665,0.291770,0.974313,0.257235,0.237207,0.364597,0.283293,1.231903,1.426030,1.944420,...,0.000845,0.060902,0.744541,0.009767,0.385967,0.054457,1.170664,1.063945,0.006799,0.496794


## Step 4: Build the image-subset dataframe and shared train/test split

In [5]:
df_subset = df[df["id"].isin(embeddings.index)].merge(
    embeddings, left_on="id", right_index=True, how="inner"
)
print(f"Subset size: {df_subset.shape[0]} listings (vs. {df.shape[0]} in the full cleaned dataset)")

image_cols = [c for c in df_subset.columns if c.startswith("img_")]

y_log = np.log1p(df_subset[TARGET_COLUMN])
train_df, test_df, y_train_log, y_test_log = train_test_split(
    df_subset, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
)


Subset size: 2894 listings (vs. 39417 in the full cleaned dataset)


## Step 5: Build both variants (baseline vs. + image PCA features) on the same subset

In [6]:
numeric_cols = df.drop(columns=[TARGET_COLUMN, "id"], errors="ignore").select_dtypes(include=[np.number]).columns

encoders = fit_categorical_encoders(train_df)
train_cat = transform_categorical_features(train_df, encoders)
test_cat = transform_categorical_features(test_df, encoders)

X_train_baseline = pd.concat([train_df[numeric_cols], train_cat], axis=1)
X_test_baseline = pd.concat([test_df[numeric_cols], test_cat], axis=1)

# PCA fit on the training split's image embeddings only (avoids leakage)
pca = PCA(n_components=30, random_state=RANDOM_STATE)
train_img_pca = pd.DataFrame(
    pca.fit_transform(train_df[image_cols]),
    columns=[f"img_pca_{i}" for i in range(30)],
    index=train_df.index,
)
test_img_pca = pd.DataFrame(
    pca.transform(test_df[image_cols]),
    columns=[f"img_pca_{i}" for i in range(30)],
    index=test_df.index,
)
print(f"Image PCA components explain {pca.explained_variance_ratio_.sum()*100:.1f}% of embedding variance")

X_train_image = pd.concat([X_train_baseline, train_img_pca], axis=1)
X_test_image = pd.concat([X_test_baseline, test_img_pca], axis=1)


Image PCA components explain 56.1% of embedding variance


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:355: RuntimeWar

## Step 6: Train XGBoost on each variant and compare

In [7]:
variants = {
    "A: baseline (numeric + categorical), image subset only": (X_train_baseline, X_test_baseline),
    "B: + image embeddings (30 PCA components)": (X_train_image, X_test_image),
}

results = {}
for name, (X_train, X_test) in variants.items():
    model = train_xgboost(X_train, y_train_log)
    preds_log = model.predict(X_test)
    metrics = print_metrics_log_target(name, y_test_log, preds_log)
    metrics["n_features"] = X_train.shape[1]
    results[name] = metrics


A: baseline (numeric + categorical), image subset only | R^2 Score: 59.24% | MAE: $271.06 | RMSE: $537.23


B: + image embeddings (30 PCA components) | R^2 Score: 57.59% | MAE: $270.79 | RMSE: $543.84


## Step 7: Summary

In [8]:
comparison = pd.DataFrame(results).T[["n_features", "mae", "rmse", "r2"]]
comparison


,n_features,mae,rmse,r2
"A: baseline (numeric + categorical), image subset only",38.0,271.063076,537.228597,0.592380
B: + image embeddings (30 PCA components),68.0,270.793326,543.841351,0.575866
